<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [1]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. Warm-up: weak prompt vs project-aware prompt

In your assistant, ask both and paste short excerpts of the answers here:

- Weak: *"add a search feature"*
- Project-aware: *"read AGENTS.md, then propose a plan to add a tags filter to search_documents in src/bootcamp_agent/tools.py — plan only, no edits"*

**Paste both responses in the cell below and note the differences.**

*"Weak: I’m tracing the existing agent/tooling flow so I can add the search feature in the right seam and verify it behaves with the repo’s patterns.

Searched for terms and reviewed 4 files
I’ve narrowed this to the repository’s actual feature request: a tags filter on the document search tool. I’m checking the document model and retrieval contract so the fix matches the existing API.

Reviewed 4 files and implemented search with tags
Ran cd /home/welela/dev3pack-cohort-2026-09 && uv run ...
Planned search feature and implemented search_documents method

Project-aware: I’m reading the repo policy and the tool contract first so the plan fits the project’s actual constraints and avoids editing anything yet.

Reviewed 2 files
Plan for the tags filter on search_documents
I read AGENTS.md and the current tool contract in tools.py. The safest plan is:

Extend search_documents to accept an optional tags argument while keeping its existing query and max_results behavior intact.
Validate at the boundary:
reject empty queries
accept either a single tag or a sequence of tags
reject empty/unknown tags with a ToolError that names the valid tag set
keep max_results clamped to MAX_SEARCH_RESULTS as the code already does
Filter the candidate documents before retrieval:
narrow documents to only items whose doc.tags include the requested tag(s)
leave the result formatting unchanged so callers still receive the same ranked passage output
Keep the change local to tools.py; no unrelated files or dependencies.
Verify with the repo’s lightweight checks: lint the file and run a focused offline probe that exercises valid and invalid tag input.
This matches the repo guidance: narrow contracts, visible validation, and read-only tool behavior.

MAI-Code-1.1-Flash • 1.2 credits"*

## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** *Rejected change: The assistant proposed validating input tags against a predefined known tag set and raising a custom ToolError if unknown tags were passed.

Reason: Out of scope and unnecessary complexity. The requirement was strictly to filter documents by tags, not to create a valid-tag registry or introduce new error types.*

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [7]:
loop = {
    "plan_approved": "Extend search_documents in tools.py with optional tags parameter and filter candidates before retrieve().",
    "diff_inspected": "Replaced 12 lines with 23 in tools.py: added tags: list[str] | None = None, filtered via any(tag in doc.tags for tag in tags) before retrieve, kept max_results slice.",
    "rejected_change": "Rejected validating tags against a known tag set and raising a custom ToolError.",
    "why_rejected": "Unnecessary and out of scope; introduced unrequested schema validation and error types.",
    "risks": "Empty tags list tags=[] bypasses filtering; uses OR semantics instead of AND; exact case-sensitive matching only; no coercion for non-list iterables.",
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      Extend search_documents in tools.py with optional tags par
diff_inspected     Replaced 12 lines with 23 in tools.py: added tags: list[st
rejected_change    Rejected validating tags against a known tag set and raisi
why_rejected       Unnecessary and out of scope; introduced unrequested schem
risks              Empty tags list tags=[] bypasses filtering; uses OR semant


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [8]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [9]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.